# Dependencies

In [ ]:
import tqdm as notebook_tqdm
from datasets import load_dataset
from dotenv import load_dotenv
from typing import Callable, Any
from pathlib import Path
import torch as T
import numpy as np
import string
import os
import matplotlib.pyplot as plt
from sklearn.decomposition import PCA

# Setup

In [2]:
# Change print width
T.set_printoptions(linewidth=1000)
np.set_printoptions(linewidth=1000)

# Data
Importing the data

In [3]:
env_path = os.path.join(os.getcwd(), ".env.secrets")
load_dotenv(env_path)

hf_token = os.getenv("HF_TOKEN")

FORMAT = "torch"

ds = load_dataset(
    "Salesforce/wikitext",
    "wikitext-103-v1",
    token=hf_token if hf_token else False,
).with_format(FORMAT)

x_test, x_train, x_val = (ds[key] for key in ds.keys())

display(x_train["text"][:10])

['',
 ' = Valkyria Chronicles III = \n',
 '',
 ' Senjō no Valkyria 3 : <unk> Chronicles ( Japanese : 戦場のヴァルキュリア3 , lit . Valkyria of the Battlefield 3 ) , commonly referred to as Valkyria Chronicles III outside Japan , is a tactical role @-@ playing video game developed by Sega and Media.Vision for the PlayStation Portable . Released in January 2011 in Japan , it is the third game in the Valkyria series . Employing the same fusion of tactical and real @-@ time gameplay as its predecessors , the story runs parallel to the first game and follows the " Nameless " , a penal military unit serving the nation of Gallia during the Second Europan War who perform secret black operations and are pitted against the Imperial unit " <unk> Raven " . \n',
 " The game began development in 2010 , carrying over a large portion of the work done on Valkyria Chronicles II . While it retained the standard features of the series , it also underwent multiple adjustments , such as making the game more forgiving

## Data pre-processing
Preprocessing the data allows us to collect as much useful information as possible. Considering that our goal is to analyze the semantic meaning of individual words and not sentences, we made some specific design choices in the preprocessing. Both of our operations were chosen on the basis that our representations do not attempt to represent sentiment, inflection, grammatical usage, or any other contextualized differences between word representations. Instead, we focus on representing and comparing the words in isolation. Therefore, we perform the following operations:

- Remove punctuation: The inclusion of punctuation would be misleading since it is a tool for expressing meaning in the context of a sentence. For instance a question, a pause, an exclamation, and so on. They are grammatical tools which do not alter the meaning of the word they are attached to. For instance, "apple!", "apple?", "apple,", and "apple." all have the word apple which we aim to represent, not the inflection of the word.

- Convert to lowercase: This keeps all words equal before their representation, removing possible differences in representation for capitalized words such as "Apple" and "apple".

# Setup Training Data

In [ ]:
class Tokenizer():
    def __init__(self, ds, num_max_sentences, window_size, PAD_TOKEN, min_count=5):
        self.ds = ds
        self.num_max_sentences = num_max_sentences
        self.window_size = window_size
        self.PAD_TOKEN = PAD_TOKEN
        self.min_count = min_count

        self.vocabulary = self.create_vocabulary(ds)
        self.PAD_INDEX = len(self.vocabulary)

        self.word_to_index = {word: i for i, word in enumerate(self.vocabulary)}
        self.word_to_index[PAD_TOKEN] = self.PAD_INDEX

        self.index_to_word = {i: word for i, word in enumerate(self.vocabulary)}
        self.index_to_word[self.PAD_INDEX] = PAD_TOKEN

        self.VOCAB_SIZE = len(self.vocabulary) + 1
        self.word_freq = self._build_freq()

        print(f"Vocabulary: {self.VOCAB_SIZE} words (min_count={min_count}, sentences={num_max_sentences})")

    def create_vocabulary(self, data):
        """Build vocabulary from training sentences only, keeping words that
        appear at least min_count times. This keeps vocab size proportional
        to the actual training data rather than the full corpus."""
        to_remove = string.punctuation + "\u201c\u201d\u2018\u2019"
        table = str.maketrans("", "", to_remove)
        freq = {}
        for sentence in data["text"][:self.num_max_sentences]:
            for word in sentence.translate(table).lower().split():
                freq[word] = freq.get(word, 0) + 1
        return sorted(w for w, c in freq.items() if c >= self.min_count)

    def _build_freq(self) -> np.ndarray:
        freq = np.zeros(self.VOCAB_SIZE, dtype=np.float32)
        for sentence in self.ds["text"][:self.num_max_sentences]:
            for w in self.preprocess_sentence(sentence):
                idx = self.word_to_index.get(w)
                if idx is not None:
                    freq[idx] += 1
        return freq

    def get_one_hot_encoding_from_token_id(self, id: int):
        vec = T.zeros(self.VOCAB_SIZE)
        vec[id] = 1
        return vec

    def get_one_hot_encoding(self, word: str):
        vec = T.zeros(self.VOCAB_SIZE)
        vec[self.word_to_index[word]] = 1
        return vec

    def get_vector_one_hot_encodings(self, words: list[str]):
        return [self.get_one_hot_encoding(word) for word in words]

    def preprocess_sentence(self, sentence: str) -> list[str]:
        to_remove = string.punctuation + "\u201c\u201d\u2018\u2019"
        table = str.maketrans("", "", to_remove)
        return sentence.translate(table).lower().split()

    def generate_CBOW_pairs(self, add_padding=False, start=0.0, end=1.0):
        """Generator that yields (context_tensor, target_id) one pair at a time.
        Words absent from the vocabulary (below min_count) are skipped.
        start/end are fractions of num_max_sentences, used for train/val splitting."""
        start_idx = int(start * self.num_max_sentences)
        end_idx   = int(end   * self.num_max_sentences)
        for sentence in self.ds["text"][start_idx:end_idx]:
            if len(sentence) < 1:
                continue
            # Only keep words that made it into the vocabulary
            ids = [self.word_to_index[w]
                   for w in self.preprocess_sentence(sentence)
                   if w in self.word_to_index]
            for center in range(len(ids)):
                context = []
                for offset in range(-self.window_size, self.window_size + 1):
                    if offset == 0:
                        continue
                    pos = center + offset
                    if 0 <= pos < len(ids):
                        context.append(ids[pos])

                if len(context) == 0:
                    continue

                if add_padding and len(context) < (2 * self.window_size):
                    dn = 2 * self.window_size - len(context)
                    context = [self.PAD_INDEX] * dn + context

                yield (T.tensor(context), ids[center])

    def create_CBOW_pairs(
            self,
            add_padding=False,
            one_hot_encoded_context=False,
            one_hot_encoded_target=False
        ):
        pairs = []
        for sentence in (self.ds["text"][:self.num_max_sentences]):
            if len(sentence) < 1:
                continue
            ids = [self.word_to_index[w]
                   for w in self.preprocess_sentence(sentence)
                   if w in self.word_to_index]
            for center in range(len(ids)):
                context = []
                for offset in range(-self.window_size, self.window_size + 1):
                    if offset == 0:
                        continue
                    pos = center + offset
                    if 0 <= pos < len(ids):
                        context.append(ids[pos])

                if len(context) == 0:
                    continue

                if add_padding and len(context) < (2 * self.window_size):
                    dn = 2 * self.window_size - len(context)
                    context = [self.PAD_INDEX] * dn + context
                target = ids[center]

                if one_hot_encoded_context:
                    context = [self.get_one_hot_encoding_from_token_id(token) for token in context]
                else:
                    context = T.tensor(context)

                if one_hot_encoded_target:
                    target = self.get_one_hot_encoding_from_token_id(target)

                pairs.append((context, target))
        return pairs

In [ ]:
FN = "/home/enarde/Documents/Design-Of-AI-Systems/miniproject/tokenizer.pt"
max_sentences = 2500
window_size   = 2
PAD_TOKEN     = "<PAD>"
min_count     = 3       # words appearing fewer than this many times are excluded

file_path = Path(FN)
if file_path.exists():
    tokenizer = T.load(file_path, weights_only=False)
    stale = (tokenizer.num_max_sentences != max_sentences
             or getattr(tokenizer, "min_count", None) != min_count)
    if stale:
        print(f"Rebuilding tokenizer (sentences {tokenizer.num_max_sentences}→{max_sentences}, min_count {getattr(tokenizer,'min_count',None)}→{min_count})")
        tokenizer = Tokenizer(x_train, max_sentences, window_size, PAD_TOKEN, min_count)
        T.save(tokenizer, file_path)
else:
    tokenizer = Tokenizer(x_train, max_sentences, window_size, PAD_TOKEN, min_count)
    T.save(tokenizer, file_path)

Rebuilding tokenizer (sentences 5000→2500, min_count 3→3)
Vocabulary: 5531 words (min_count=3, sentences=2500)


In [ ]:
class DataLoader():
    def __init__(self, data, batch_size, shuffle=False):
        self.data = data
        self.batch_size = batch_size
        self.shuffle = shuffle
        self._indices = None
        self.num_data_points = len(data)

    def __iter__(self):
        indices = list(range(len(self.data)))
        if self.shuffle:
            np.random.shuffle(indices)
        self._indices = indices
        self._pos = 0
        return self

    def __next__(self):
        if self._pos >= len(self.data):
            raise StopIteration
        batch = [self.data[i] for i in self._indices[self._pos : self._pos + self.batch_size]]
        self._pos += self.batch_size
        return batch

    def __len__(self):
        return len(self.data) // self.batch_size


class StreamingDataLoader():
    """DataLoader that generates (context, target) pairs on demand via a generator
    function. No list of all pairs is ever held in memory."""

    def __init__(self, pair_fn, batch_size):
        self.pair_fn = pair_fn      # callable: () -> generator of (context, target_id)
        self.batch_size = batch_size
        # Count pairs once at construction so len() and num_data_points are available
        self.num_data_points = sum(1 for _ in pair_fn())

    def __len__(self):
        return self.num_data_points // self.batch_size

    def __iter__(self):
        batch = []
        for pair in self.pair_fn():
            batch.append(pair)
            if len(batch) == self.batch_size:
                yield batch
                batch = []
        if batch:   # yield last partial batch
            yield batch


TRAIN_SPLIT = 0.8

dataloader       = StreamingDataLoader(lambda: tokenizer.generate_CBOW_pairs(add_padding=True, end=TRAIN_SPLIT),batch_size=32)
dataloader_val   = StreamingDataLoader(lambda: tokenizer.generate_CBOW_pairs(add_padding=True, start=TRAIN_SPLIT),batch_size=32)
dataloader_ns    = StreamingDataLoader(lambda: tokenizer.generate_CBOW_pairs(add_padding=True, end=TRAIN_SPLIT),batch_size=32)
dataloader_ns_val= StreamingDataLoader(lambda: tokenizer.generate_CBOW_pairs(add_padding=True, start=TRAIN_SPLIT),batch_size=32)

print(f"Train pairs: {dataloader.num_data_points}  |  Val pairs: {dataloader_val.num_data_points}")

Train pairs: 91763  |  Val pairs: 24467


# Creating the modeling tools
We chose to define our model in an object oriented way. The reason for this being that we aim to increase the readability of our code by relating python objects to mathematical objects. Hopefully, creating a successfull bridge between the theory and the implementation.

Each layer of our model can be modeled as a function.
$$
        \mathbf{\hat{y}} = f(\mathbf{v})
$$
This function can contain any arbitrary operation on the input vector given that it returns a Tensor object as a result. This effectively functions as an interface for clients to use.

In [ ]:
class Layer():
    def forward(self, v: Any) -> T.Tensor:
        raise NotImplementedError(f"{type(self).__name__} has not implemented forward")

In [ ]:
class Derivable(Layer):
    """A layer that can compute gradients but has no trainable weights (e.g. activations)."""

    def __init__(self):
        super().__init__()
        self.dLdv: T.Tensor = None  # gradient of loss w.r.t. this layer's input

    def backward(self, dLdy: T.Tensor):
        """Compute and store dLdv given upstream gradient dLdy."""
        raise NotImplementedError(f"{type(self).__name__} has not implemented backward")

In [ ]:
class Trainable(Derivable):
    """A layer with trainable weights supporting mini-batch SGD.

    Mini-batch training protocol:
      1. model.zero_grad()                      — clear accumulators (once per batch)
      2. model.accumulate_gradients(dLdy)        — per sample, inside the batch loop
      3. model.backpropogate(batch_size)         — apply weight update (once per batch)
    """

    def __init__(self, lr):
        super().__init__()
        self.lr:   float    = lr
        self.dLdw: T.Tensor = None  # gradient of loss w.r.t. weights
        self.dLdv: T.Tensor = None  # gradient of loss w.r.t. input

    def zero_grad(self) -> None:
        """Clear gradient accumulators before a new batch."""
        raise NotImplementedError(f"{type(self).__name__} has not implemented zero_grad")

    def accumulate_gradients(self, dLdy: T.Tensor) -> None:
        """Compute gradients for one sample and add to accumulators.
        Sets dLdv for the chain rule. Does NOT update weights."""
        raise NotImplementedError(f"{type(self).__name__} has not implemented accumulate_gradients")

    def update_weights(self, batch_size: int) -> None:
        """Apply accumulated gradients divided by batch_size, then reset accumulators.
        Called once per batch after all accumulate_gradients calls."""
        raise NotImplementedError(f"{type(self).__name__} has not implemented update_weights")

## One-hot-encoder
Assigns each word in a vocabulary a $n$ dimensional vector with a single element set to one (uniquely) where the words are sorted alphabetically and $n$ is the number of unique words

In [163]:
class OneHotEncoder(Layer):
    def __init__(self, vocabulary_size: int):
        self.dim = vocabulary_size

    def forward(self, token: int) -> T.Tensor:
        vec = T.zeros(self.dim)
        vec[token] = 1
        return vec
    

## Vectorized One-hot-encoder

In [164]:
class OneHotEncoderVectorized(Layer):
    def __init__(self, encoder: OneHotEncoder):
        self.encoder = encoder

    def forward(self, tokens: list[int]) -> T.Tensor:
        return T.stack([self.encoder.forward(token) for token in tokens])

In [165]:
FN = "/home/enarde/Documents/Design-Of-AI-Systems/miniproject/one_hot_encoder.pt"
file_path = Path(FN)
if file_path.exists():
    encoder = T.load(file_path, weights_only=False)
else:
    encoder = OneHotEncoder(tokenizer.VOCAB_SIZE)
    T.save(encoder, file_path)

## Softmax
Softmax is a function that turns a vector of numbers $\mathbf{z} \in \mathbf{R}^n$ in to a vector of probilities $\mathbf{\hat{y}}$ s.t $\sum_i \hat{y}_i = 1$. The softmax function is defined as:

$$\mathbf{\hat{y}}_i = \frac{e^{z_i}}{\sum_{j=1}^{n} e^{z_j}}$$

The gradients are calculated automatically during the backward pass and are set as follows:

$$
    \begin{align*}
        & \delta_{ij} = \begin{cases} 1 & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases} \\
        & \frac{\partial \hat{y}_i}{\partial z_j} = \hat{y}_i(\delta_{ij} - \hat{y}_j) \\
        & \text{diag}(\mathbf{\hat{y}}) = \begin{bmatrix} \hat{y}_1 & 0 & \dots \\ 0 & \hat{y}_2 & \dots \\ \vdots & \vdots & \ddots \end{bmatrix}
        & \mathbf{\hat{y}}\mathbf{\hat{y}}^T = \begin{bmatrix} \hat{y}_1^2 & \hat{y}_1\hat{y}_2 & \dots \\ \hat{y}_2\hat{y}_1 & \hat{y}_2^2 & \dots \\ \vdots & \vdots & \ddots \end{bmatrix}\\
        & \frac{\partial{\mathbf{\hat{y}}}}{\partial{\mathbf{z}}} = \text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T
    \end{align*}
$$

The single element version $\frac{\partial \hat{y}_i}{\partial z_j}$ of the derivative is included for clarity as it makes it easier to understand the full vector ($\mathbf{\hat{y}}$). 

In order to calculate the partial derivative of the loss given the input $\mathbf{z}$ to the activation layer we can apply the upstream partial derivative denoted $\frac{\partial{L}}{\partial{\mathbf{\hat{y}}}}$ using the chain rule:

$$
    \begin{align*}
        & \frac{\partial{L}}{\partial{\mathbf{z}}} = \left(\frac{\partial{\mathbf{\hat{y}}}}{\partial{\mathbf{z}}}\right)^T \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{z}}} = (\text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T) \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{z}}} = \text{diag}(\mathbf{\hat{y}}) \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} - \mathbf{\hat{y}}\mathbf{\hat{y}}^T \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{z}}} = \mathbf{\hat{y}}\odot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} - \left(\mathbf{\hat{y}}^T \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}}\right)\mathbf{\hat{y}} \\
    \end{align*}
$$
                
Where the first transposition is required to match the dimensionalities since we are working with column vectors. Furthermore, the last line follows from simplifications of the diagonal matrix multiplied by a vector reducing to an elementwise scale of the upstream partial derivative and the second line results from the dot product reducing the vector multiplication to a scalar multiplication. Note that $\left(\text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T\right)^T = \text{diag}(\mathbf{\hat{y}}) - \mathbf{\hat{y}}\mathbf{\hat{y}}^T$ due to symmetry.

In [234]:
class Softmax(Derivable):
    def __init__(self):
        super().__init__()
        self._cached_output = None

    def forward(self, z: T.Tensor) -> T.Tensor:
        # Exponentiate all elements of z, z are the logits, or the "raw" input.
        z_exp = T.exp(z)
        # Squeeze away the last element of y, this makes no difference mathematically.
        y = (z_exp / T.sum(z_exp, dim=0)).squeeze()

        self._cached_output = y
        return y
    
    def compute_gradients(self, dLdy: T.Tensor):
        return (self._cached_output * (dLdy - T.dot(self._cached_output, dLdy)))
    
    def backward(self, dLdy: T.Tensor):
        self.dLdv = self.compute_gradients(dLdy)

sm = Softmax()
test_values = [
    T.tensor([1.0, 0.0, 0.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 0.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 1.0, 0.0]).reshape(-1, 1),
    T.tensor([1.0, 1.0, 1.0, 1.0]).reshape(-1, 1),
]

for v in test_values:
    out = sm.forward(v)
    print(
        f"""Output:\n{out.numpy()},
        """
    )

Output:
[0.4753669 0.1748777 0.1748777 0.1748777],
        
Output:
[0.3655293  0.3655293  0.13447072 0.13447072],
        
Output:
[0.29692274 0.29692274 0.29692274 0.10923178],
        
Output:
[0.25 0.25 0.25 0.25],
        


## Linear layer
The object `LinearLayer2D` performs the following operation on an input vector $\mathbf{v}$:

$$
    \begin{align*}
        & \mathbf{z} = \mathbf{v}^T \mathbf{W} + \mathbf{b} \\
        & \mathbf{\hat{y}} = \sigma{(\mathbf{z})}
    \end{align*}
$$

Where $\mathbf{b}$ is the bias term, $\mathbf{W}$ is the weights, and $\mathbf{\sigma}$ is the activation function, $\mathbf{z}$ is just an intermediary step used for convenience.

We have, however, modified this operation slightly so that we can remove the separate bias term and include it directly in an expanded weight matrix $\mathbf{W_b}$. This is achieved by adding a row filled with ones to $\mathbf{W}$. So the operation actually becomes:

$$
    \begin{align*}
        & \mathbf{z} = \mathbf{v}^T \mathbf{W_b} \\
        & \mathbf{\hat{y}} = \sigma{(\mathbf{z})}
    \end{align*}
$$

The gradients are calculated as following during the backward pass:

**$\mathbf{\hat{y}}$ w.r.t the input vector $\mathbf{v}$**
$$
    \begin{align*}
        & z_i = v_1 \mathbf{W_b}_{1i} + v_2 \mathbf{W_b}_{2i} + \dots + v_i \mathbf{W_b}_{ii} + \dots + v_N \mathbf{W_b}_{ni} \\

        & \frac{\partial \mathbf{z_i}}{\partial \mathbf{v_j}} = \mathbf{W_b}_{ji} \\

        & \frac{\partial \mathbf{z}}{\partial \mathbf{v}} = \begin{bmatrix} \frac{\partial z_1}{\partial v_1} & \frac{\partial z_1}{\partial v_2} & \dots & \frac{\partial z_1}{\partial v_N} \\ \frac{\partial z_2}{\partial v_1} & \frac{\partial z_2}{\partial v_2} & \dots & \frac{\partial z_2}{\partial v_N} \\ \vdots & \vdots & \ddots & \vdots \\ \frac{\partial z_M}{\partial v_1} & \frac{\partial z_M}{\partial v_2} & \dots & \frac{\partial z_M}{\partial v_N} \end{bmatrix} = \mathbf{W_b}^T \\

        & \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \text{ is taken from the activation function with} \space \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \space \text{applied} \\

        & \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{v}} = \frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}} \cdot \mathbf{W_b}^T        
    \end{align*}
$$

Therefore, we can calculate $\frac{\partial{L}}{\partial{\mathbf{v}}}$ with:

$$
    \begin{align*}
        & \frac{\partial{L}}{\partial{\mathbf{v}}} = \left(\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{v}}\right)^T \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{v}}} = \left((\mathbf{W_b}^T)^T \cdot \left(\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}}\right)^T \right) \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{v}}} = \mathbf{W_b} \cdot \left(\frac{\partial \mathbf{\hat{y}}}{\partial \mathbf{z}}\right)^T \cdot \frac{\partial{L}}{\partial{\mathbf{\hat{y}}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{v}}} = \mathbf{W_b} \cdot \frac{\partial{L}}{\partial{\mathbf{z}}} \\
    \end{align*}
$$

**$\mathbf{\hat{y}}$ w.r.t the weight matrix $\mathbf{W_b}$**

$$
    \begin{align*}
        & z_i = v_1 \mathbf{W_b}_{1i} + v_2 \mathbf{W_b}_{2i} + \dots + v_i \mathbf{W_b}_{ii} + \dots + v_N \mathbf{W_b}_{ni} + 1 \cdot \mathbf{W_b}_{(n+1)i} \\

        & \frac{\partial z_i}{\partial \mathbf{W_b}_{kj}} = \begin{cases} v_k & \text{if } i = j \\ 0 & \text{if } i \neq j \end{cases} \\

        & \frac{\partial \mathbf{z}}{\partial \mathbf{W_b}} = \left[ \quad \begin{bmatrix} v_1 & 0 & \dots & 0 \\ v_2 & 0 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ v_N & 0 & \dots & 0 \\ 1 & 0 & \dots & 0 \end{bmatrix} \quad \Bigg| \quad \begin{bmatrix} 0 & v_1 & \dots & 0 \\ 0 & v_2 & \dots & 0 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & v_N & \dots & 0 \\ 0 & 1 & \dots & 0 \end{bmatrix} \quad \Bigg| \quad \dots \quad \Bigg| \quad \begin{bmatrix} 0 & 0 & \dots & v_1 \\ 0 & 0 & \dots & v_2 \\ \vdots & \vdots & \ddots & \vdots \\ 0 & 0 & \dots & v_N \\ 0 & 0 & \dots & 1 \end{bmatrix} \quad \right]
    \end{align*}
$$

Therefore, we can calculate $\frac{\partial{L}}{\partial{\mathbf{W_b}}}$ where $\frac{\partial{L}}{\partial{\mathbf{z}}}$ is our upstream gradient. We can do this by applying the chain rule element-wise:
$$
\begin{align*}
& \frac{\partial L}{\partial \mathbf{W_b}_{ij}} = \frac{\partial L}{\partial z_j} \cdot \frac{\partial z_j}{\partial \mathbf{W_b}_{ij}} = \frac{\partial L}{\partial z_j} \cdot v_{b,i}
\end{align*}
$$

This is equivalent to the outer product:
$$
    \begin{align*}
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \left(\frac{\partial L}{\partial \mathbf{z}}\right)^T \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \begin{bmatrix} v_1 \\ v_2 \\ \vdots \\ v_N \\ 1 \end{bmatrix} \begin{bmatrix} \frac{\partial L}{\partial z_1} & \frac{\partial L}{\partial z_2} & \dots & \frac{\partial L}{\partial z_M} \end{bmatrix} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \begin{bmatrix} 
        v_1 \frac{\partial L}{\partial z_1} & v_1 \frac{\partial L}{\partial z_2} & \dots & v_1 \frac{\partial L}{\partial z_M} \\ 
        v_2 \frac{\partial L}{\partial z_1} & v_2 \frac{\partial L}{\partial z_2} & \dots & v_2 \frac{\partial L}{\partial z_M} \\ 
        \vdots & \vdots & \ddots & \vdots \\ 
        v_N \frac{\partial L}{\partial z_1} & v_N \frac{\partial L}{\partial z_2} & \dots & v_N \frac{\partial L}{\partial z_M} \\ 
        \frac{\partial L}{\partial z_1} & \frac{\partial L}{\partial z_2} & \dots & \frac{\partial L}{\partial z_M} 
        \end{bmatrix} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \otimes \frac{\partial L}{\partial \mathbf{z}}
    \end{align*}
$$

<!-- Therefore, we can calculate $\frac{\partial{L}}{\partial{\mathbf{W_b}}}$ where $\frac{\partial{L}}{\partial{\mathbf{z}}}$ is our upstream gradient which can contain the application of an activation or not. This becomes:

$$
    \begin{align*}
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \left(\frac{\partial \mathbf{z}}{\partial \mathbf{W_b}}\right)^T \cdot \frac{\partial{L}}{\partial{\mathbf{z}}} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \left(\frac{\partial L}{\partial \mathbf{z}}\right)^T \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \begin{bmatrix} v_{b,1} \\ v_{b,2} \\ \vdots \\ v_{b,N} \\ 1 \end{bmatrix} \begin{bmatrix} \frac{\partial L}{\partial z_1} & \frac{\partial L}{\partial z_2} & \dots & \frac{\partial L}{\partial z_M} \end{bmatrix} = \begin{bmatrix} v_{b,1} \frac{\partial L}{\partial z_1} & v_{b,1} \frac{\partial L}{\partial z_2} & \dots & v_{b,1} \frac{\partial L}{\partial z_M} \\ v_{b,2} \frac{\partial L}{\partial z_1} & v_{b,2} \frac{\partial L}{\partial z_2} & \dots & v_{b,2} \frac{\partial L}{\partial z_M} \\ \vdots & \vdots & \ddots & \vdots \\ v_{b,N} \frac{\partial L}{\partial z_1} & v_{b,N} \frac{\partial L}{\partial z_2} & \dots & v_{b,N} \frac{\partial L}{\partial z_M} \\ \frac{\partial L}{\partial z_1} & \frac{\partial L}{\partial z_2} & \dots & \frac{\partial L}{\partial z_M} \end{bmatrix}
    \end{align*}
$$

Therefore, we can calculate $\frac{\partial{L}}{\partial{\mathbf{W_b}}}$ with:

$$
    \begin{align*}
        & \frac{\partial L}{\partial \mathbf{W_b}_{ij}} = \frac{\partial L}{\partial z_j} \cdot v_{b,i} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \otimes \frac{\partial L}{\partial \mathbf{z}} \\
        & \frac{\partial{L}}{\partial{\mathbf{W_b}}} = \mathbf{v_b} \left(\frac{\partial L}{\partial \mathbf{z}}\right)^T
    \end{align*}
$$ -->

In [ ]:
class LinearLayer2D(Trainable):
    def __init__(self, dim_in, dim_out, activation, lr, is_layer_in_average=False):
        super().__init__(lr=lr)
        self.dim_in  = dim_in
        self.dim_out = dim_out
        self.weights = T.randn(dim_in + 1, dim_out) * 0.01
        self.activation = activation
        self.is_layer_in_average = is_layer_in_average
        self._cached_input = []
        self._dLdw_accum   = T.zeros(dim_in + 1, dim_out)  # gradient accumulator

    def __add_bias(self, v):
        return T.cat((v, T.ones((1,))), dim=0)

    def forward(self, v):
        vec_plus_bias = self.__add_bias(v)
        self._cached_input.append(vec_plus_bias)
        z = vec_plus_bias @ self.weights
        return self.activation.forward(z).squeeze() if self.activation else z.squeeze()

    def accumulate_gradients(self, dLdy):
        """Compute gradient for this sample and accumulate. Does NOT update weights."""
        if not self._cached_input:
            raise RuntimeError("Perform a forward pass before accumulating gradients")
        if self.activation is not None:
            self.activation.backward(dLdy)
            dLdz = self.activation.dLdv
        else:
            dLdz = dLdy
        cached_input = self._cached_input.pop()
        self.dLdv          = (self.weights @ dLdz)[:-1]   # strip bias row
        self._dLdw_accum  += T.outer(cached_input, dLdz)

    def zero_grad(self):
        self._dLdw_accum.zero_()

    def update_weights(self, batch_size):
        """Apply accumulated gradient and reset. Call once per batch."""
        self.weights      -= self.lr * self._dLdw_accum / batch_size
        self._dLdw_accum.zero_()


test_values = [T.ones((10,), dtype=T.float32)]
ll = LinearLayer2D(10, 3, Softmax(), 1e-3)
for v in test_values:
    out = ll.forward(v)
    print(f"Input: {v.numpy()}\nOutput: {out.numpy()}\n")

Input: [1. 1. 1. 1. 1. 1. 1. 1. 1. 1.]
Output: [0.3376489  0.33382863 0.32852256]



## Linear layer average mixer
This object is used to average the the result of many layers of the same type.

In [ ]:
class LinearLayerAvgMixer(Trainable):

    def __init__(self, n: int, layer: LinearLayer2D, lr=None):
        super().__init__(lr)
        self.n     = n
        self.layer = layer

    def forward(self, v: T.Tensor) -> T.Tensor:
        avg = T.zeros(self.layer.dim_out)
        for i in range(self.n):
            avg += self.layer.forward(v[i]) / self.n
        return avg

    def zero_grad(self) -> None:
        self.layer.zero_grad()

    def accumulate_gradients(self, dLdy: T.Tensor) -> None:
        for _ in range(self.n):
            self.layer.accumulate_gradients(dLdy / self.n)
        self.dLdv = self.layer.dLdv

    def update_weights(self, batch_size: int) -> None:
        self.layer.update_weights(batch_size)

## Embedding Average Layer (Optimized Lookup)

The `LinearLayerAvgMixer` averages context embeddings by:
1. Encoding each context word index as a one-hot vector $e_i \in \mathbb{R}^V$
2. Multiplying by the weight matrix: $e_i W = W_{[i]}$ (row $i$ of $W$)
3. Averaging the results

Step 1 and 2 together are equivalent to a direct row lookup, since multiplying a one-hot vector by a matrix simply selects the corresponding row:

$$\frac{1}{n}\sum_{k=1}^{n} e_{i_k} W = \frac{1}{n}\sum_{k=1}^{n} W_{[i_k]}$$

`EmbeddingAverageLayer` implements this directly without large matrix multiplication and memory costly allocations. It is a mathematically equivalent simplification used to speed up training.

In [ ]:
class EmbeddingAverageLayer(Trainable):
    """
    Optimized replacement for OneHotEncoderVectorized + LinearLayerAvgMixer.

    Mathematically equivalent to averaging one-hot context vectors multiplied by W,
    but replaces the (vocab_size x embed_dim) matmul with a direct row lookup:

        (1/n) * sum(e_{i_k} @ W)  ==  (1/n) * sum(W[i_k])

    This avoids allocating large one-hot vectors (vocab_size-dim) entirely.
    """

    def __init__(self, vocab_size: int, embed_dim: int, lr: float = 1e-3):
        super().__init__(lr=lr)
        self.vocab_size      = vocab_size
        self.embed_dim       = embed_dim
        self.weights         = T.randn(vocab_size, embed_dim) * 0.1
        self._cached_indices = None
        self._grad_accum     = {}   # idx (int) -> accumulated gradient tensor

    def forward(self, indices: T.Tensor) -> T.Tensor:
        self._cached_indices = indices
        return self.weights[indices].mean(dim=0)   # (embed_dim,)

    def zero_grad(self) -> None:
        self._grad_accum.clear()

    def accumulate_gradients(self, dLdy: T.Tensor) -> None:
        """Accumulate gradient contribution from one sample. Does NOT update weights."""
        n    = len(self._cached_indices)
        grad = dLdy / n
        for idx in self._cached_indices:
            idx = int(idx)
            if idx in self._grad_accum:
                self._grad_accum[idx] += grad
            else:
                self._grad_accum[idx] = grad.clone()
        self.dLdv = None   # no differentiable layer upstream

    def update_weights(self, batch_size: int) -> None:
        """Apply accumulated gradients averaged over the batch, then reset."""
        for idx, grad in self._grad_accum.items():
            self.weights[idx] -= self.lr * grad / batch_size
        self._grad_accum.clear()

## The Model object
This object holds all the layers and activation functions we have defined, it represents the implementation of the full function that produces and outputs $\mathbf{\hat{y}}$ from an input $\mathbf{v}$.

In [ ]:
class Model:
    def __init__(self, layers: list[Trainable | Derivable | object], debug_mode=False):
        self.layers     = layers
        self.debug_mode = debug_mode

    def forward(self, v: T.Tensor):
        curr = v
        for l in self.layers:
            if self.debug_mode:
                print(f"Input to {type(l).__name__}: shape {curr.shape if isinstance(curr, T.Tensor) else '?'}")
            curr = l.forward(curr)
            if self.debug_mode:
                print(f"Output from {type(l).__name__}: shape {curr.shape}")
        return curr

    def zero_grad(self) -> None:
        """Clear all gradient accumulators. Call once before each batch."""
        for l in self.layers:
            if isinstance(l, Trainable):
                l.zero_grad()

    def accumulate_gradients(self, dLdy: T.Tensor) -> None:
        """Chain the gradient backward through all layers for one sample.
        Calls accumulate_gradients on each Trainable layer"""
        for l in reversed(self.layers):
            if isinstance(l, Trainable):
                l.accumulate_gradients(dLdy)
            if not isinstance(l, OneHotEncoderVectorized):
                dLdy = l.dLdv

    def backpropogate(self, batch_size: int) -> None:
        """Apply accumulated gradients to all layers. Call once per batch."""
        for l in self.layers:
            if isinstance(l, Trainable):
                l.update_weights(batch_size)

    def clear_caches(self) -> None:
        """Discard cached forward-pass inputs after a validation forward pass."""
        for l in self.layers:
            if hasattr(l, '_cached_input'):
                l._cached_input.clear()
            if hasattr(l, 'layer') and hasattr(l.layer, '_cached_input'):
                l.layer._cached_input.clear()

## Binary cross entropy loss
The Binary cross entropy loss function is a loss function that is primarily used to compare probability vectors (vectors where elements sum to $1$) against binary vectors (where a single element is set to $1$ and there rest are set to $0$.) This is usally used in binary classification where a model predicts a probability of a sample having some label, for example, $cat$, $dog$, $fish$. In binary classification is only a single correct lable but to improve model performance we allow it to be kind of uncertain, so it can predict $90\%$ $dog$ , $5\%$ $cat$, $5\%$ $fish$.
$$
    L = -\big(y \log(\hat{y}) + (1 - y) \log(1 - \hat{y})\big)
$$


In [238]:
class BinaryCrossEntropyLoss():
    def __init__(self):
        self.loss = None
        self.dLdy = None

    def compute_loss(self, y_true: T.Tensor, y_hat: T.Tensor):
        # Log of 0 gives inf. Clamping to avoid crashes
        y_hat = T.clamp(y_hat, min=1e-7, max=1 - 1e-7)

        loss = -(y_true @ T.log(y_hat) + (1-y_true) @ (T.log(1-y_hat)))
        self.loss = loss

        dLdy_hat = ((-y_true) / y_hat) + ((1-y_true) / (1-y_hat))
        self.dLdy = dLdy_hat
        return loss
        

## Negative Sampling Loss

Instead of computing softmax over the full vocabulary, negative sampling reformulates the problem as binary classification. For a context vector $\mathbf{h}$ and target word index $p$, the loss is:

$$L = -\log\sigma(\mathbf{h} \cdot \mathbf{w}_p) - \sum_{i=1}^{k} \log(1 - \sigma(\mathbf{h} \cdot \mathbf{w}_{n_i}))$$

Where $\mathbf{w}_p$ is the output embedding of the positive (target) word, $\mathbf{w}_{n_i}$ are the output embeddings of $k$ randomly sampled negative words, and $\sigma$ is the sigmoid function.

The gradients are:

$$
\begin{align*}
    & \frac{\partial L}{\partial \mathbf{w}_p} = (\sigma(\mathbf{h} \cdot \mathbf{w}_p) - 1) \cdot \mathbf{h} \\
    & \frac{\partial L}{\partial \mathbf{w}_{n_i}} = \sigma(\mathbf{h} \cdot \mathbf{w}_{n_i}) \cdot \mathbf{h} \\
    & \frac{\partial L}{\partial \mathbf{h}} = (\sigma(\mathbf{h} \cdot \mathbf{w}_p) - 1) \cdot \mathbf{w}_p + \sum_{i=1}^{k} \sigma(\mathbf{h} \cdot \mathbf{w}_{n_i}) \cdot \mathbf{w}_{n_i}
\end{align*}
$$

Negatives are sampled from a smoothed unigram distribution $P(w) \propto f(w)^{0.75}$ where $f(w)$ is the word frequency. The smoothing reduces the dominance of very frequent words.


In [ ]:
class NegativeSamplingLoss():
    def __init__(self, tokenizer, embed_dim, k=5, lr=1e-3, power=0.75):
        self.k              = k
        self.lr             = lr
        self.vocab_size     = tokenizer.VOCAB_SIZE
        self.out_embeddings = T.randn(self.vocab_size, embed_dim) * 0.1
        self.noise_dist     = self._build_noise_dist(tokenizer.word_freq, power)
        self._cached_h          = None
        self._cached_pos_id     = None
        self._cached_neg_ids    = None
        self._cached_pos_score  = None
        self._cached_neg_scores = None
        self.dLdv               = None
        self._pos_grad_accum    = {}   # pos_id -> accumulated gradient
        self._neg_grad_accum    = {}   # neg_id -> accumulated gradient

    def _build_noise_dist(self, word_freq, power):
        smoothed = word_freq ** power
        smoothed += 1e-10
        return smoothed / smoothed.sum()

    def _sample_negatives(self, exclude_id):
        samples = []
        while len(samples) < self.k:
            c = int(np.random.choice(self.vocab_size, p=self.noise_dist))
            if c != exclude_id:
                samples.append(c)
        return samples

    def compute_loss(self, h, pos_id):
        pos_vec   = self.out_embeddings[pos_id]
        pos_score = T.sigmoid(T.dot(h, pos_vec))
        pos_loss  = T.log(pos_score + 1e-7)

        neg_ids    = self._sample_negatives(pos_id)
        neg_scores = [T.sigmoid(T.dot(h, self.out_embeddings[i])) for i in neg_ids]
        neg_loss   = sum(T.log(1 - s + 1e-7) for s in neg_scores)

        self._cached_h          = h
        self._cached_pos_id     = pos_id
        self._cached_neg_ids    = neg_ids
        self._cached_pos_score  = pos_score
        self._cached_neg_scores = neg_scores
        return -(pos_loss + neg_loss)

    def zero_grad(self) -> None:
        self._pos_grad_accum.clear()
        self._neg_grad_accum.clear()

    def backward(self) -> None:
        """Compute dLdv for chain rule and accumulate output embedding gradients.
        Does NOT update weights"""
        h         = self._cached_h
        pos_id    = self._cached_pos_id
        pos_score = self._cached_pos_score

        # Gradient w.r.t. h — passed back to input embedding layer via dLdv
        dLdh = (pos_score - 1) * self.out_embeddings[pos_id].clone()

        pos_grad = (pos_score - 1) * h
        if pos_id in self._pos_grad_accum:
            self._pos_grad_accum[pos_id] += pos_grad
        else:
            self._pos_grad_accum[pos_id] = pos_grad.clone()

        for neg_id, neg_score in zip(self._cached_neg_ids, self._cached_neg_scores):
            dLdh += neg_score * self.out_embeddings[neg_id].clone()
            neg_grad = neg_score * h
            if neg_id in self._neg_grad_accum:
                self._neg_grad_accum[neg_id] += neg_grad
            else:
                self._neg_grad_accum[neg_id] = neg_grad.clone()

        self.dLdv = dLdh

    def update_weights(self, batch_size: int) -> None:
        """Apply accumulated gradients averaged over the batch, then reset."""
        for idx, grad in self._pos_grad_accum.items():
            self.out_embeddings[idx] -= self.lr * grad / batch_size
        for idx, grad in self._neg_grad_accum.items():
            self.out_embeddings[idx] -= self.lr * grad / batch_size
        self._pos_grad_accum.clear()
        self._neg_grad_accum.clear()

# Creating the Word2Vec model

In [240]:
INPUT_DIM = encoder.dim
OUTPUT_DIM = encoder.dim
LATENT_DIM = 10

word2vecCBOW = Model(
    [
        OneHotEncoderVectorized(
            encoder=encoder
        ),  # Use the saved encoder!
        LinearLayerAvgMixer(
            n=4,
            layer=LinearLayer2D(
                dim_in=INPUT_DIM, 
                dim_out=LATENT_DIM, 
                activation=None,
                lr=1e-3,
                is_layer_in_average=True
            ),
        ),
        LinearLayer2D(
            dim_in=LATENT_DIM, 
            dim_out=OUTPUT_DIM, 
            activation=Softmax(),
            lr=1e-3,
            is_layer_in_average=False
        ),
    ],
    debug_mode=False
)

## Negative Sampling Word2Vec

In [249]:
INPUT_DIM  = encoder.dim
LATENT_DIM = 10
K_NEGATIVES = 5

word2vecCBOW_ns = Model(
    [
        OneHotEncoderVectorized(encoder=encoder),
        LinearLayerAvgMixer(
            n=4,
            layer=LinearLayer2D(
                dim_in=INPUT_DIM,
                dim_out=LATENT_DIM,
                activation=None,
                lr=1e-3,
                is_layer_in_average=True
            ),
        ),
    ],
    debug_mode=False
)

ns_loss = NegativeSamplingLoss(
    tokenizer=tokenizer,
    embed_dim=LATENT_DIM,
    k=K_NEGATIVES,
    lr=1e-3
)

# Training, Validation, Testing

## Training harness
An object that trains a Model object using mini batch gradient descent

In [ ]:
def train(model, dataloader, val_dataloader, loss_fn, tokenizer, n_epochs):
    train_losses = []
    val_losses   = []

    for epoch in range(n_epochs):
        # --- Training ---
        epoch_train_loss = 0
        pbar = notebook_tqdm.tqdm(dataloader, total=len(dataloader), desc=f"Epoch {epoch+1}/{n_epochs}", leave=True)
        for batch in pbar:
            model.zero_grad()
            batch_loss = 0

            for (context, target) in batch:
                target_one_hot = tokenizer.get_one_hot_encoding_from_token_id(int(target))
                y_hat = model.forward(context)
                loss  = loss_fn.compute_loss(target_one_hot, y_hat)
                batch_loss       += loss
                model.accumulate_gradients(loss_fn.dLdy)   # accumulate per sample

            model.backpropogate(len(batch))                # weight update once per batch
            epoch_train_loss += batch_loss
            pbar.set_postfix({"train_loss": f"{float(epoch_train_loss):.4f}"})

        epoch_train_loss /= dataloader.num_data_points
        train_losses.append(epoch_train_loss)

        # --- Validation (no weight updates) ---
        epoch_val_loss = 0
        for batch in val_dataloader:
            for (context, target) in batch:
                target_one_hot = tokenizer.get_one_hot_encoding_from_token_id(int(target))
                y_hat = model.forward(context)
                model.clear_caches()
                epoch_val_loss += loss_fn.compute_loss(target_one_hot, y_hat)

        epoch_val_loss /= val_dataloader.num_data_points
        val_losses.append(epoch_val_loss)

        print(f"Epoch {epoch+1}/{n_epochs}: train={float(epoch_train_loss):.4f}  val={float(epoch_val_loss):.4f}")

    return {"train": train_losses, "val": val_losses}

In [ ]:
def train_ns(model, ns_loss, dataloader, val_dataloader, n_epochs):
    train_losses = []
    val_losses   = []

    for epoch in range(n_epochs):
        # --- Training ---
        epoch_train_loss = 0
        pbar = notebook_tqdm.tqdm(dataloader, total=len(dataloader), desc=f"Epoch {epoch+1}/{n_epochs}", leave=True)
        for batch in pbar:
            model.zero_grad()
            ns_loss.zero_grad()
            batch_loss = 0

            for (context, target_id) in batch:
                target_id = int(target_id)
                h    = model.forward(context)
                loss = ns_loss.compute_loss(h, target_id)
                batch_loss += loss.item()
                ns_loss.backward()                          # accumulate per sample
                model.accumulate_gradients(ns_loss.dLdv)   # accumulate per sample

            ns_loss.update_weights(len(batch))              # weight update once per batch
            model.backpropogate(len(batch))                 # weight update once per batch

            epoch_train_loss += batch_loss
            pbar.set_postfix({"train_loss": f"{epoch_train_loss:.4f}"})

        epoch_train_loss /= dataloader.num_data_points
        train_losses.append(epoch_train_loss)

        # --- Validation (no weight updates) ---
        epoch_val_loss = 0
        for batch in val_dataloader:
            for (context, target_id) in batch:
                target_id = int(target_id)
                h    = model.forward(context)
                model.clear_caches()
                epoch_val_loss += ns_loss.compute_loss(h, target_id).item()

        epoch_val_loss /= val_dataloader.num_data_points
        val_losses.append(epoch_val_loss)

        print(f"Epoch {epoch+1}/{n_epochs}: train={epoch_train_loss:.4f}  val={epoch_val_loss:.4f}")

    return {"train": train_losses, "val": val_losses}

## Continous bag of words

In [ ]:
ce = BinaryCrossEntropyLoss()
softmax_history = train(word2vecCBOW, dataloader, dataloader_val, ce, tokenizer, 10)

In [ ]:
ns_history = train_ns(word2vecCBOW_ns, ns_loss, dataloader_ns, dataloader_ns_val, 10)

Epoch 1/10:  80%|████▊ | 349/437 [04:43<01:11,  1.23it/s, train_loss=46446.2136]


KeyboardInterrupt: 

## Optimized Word2Vec with EmbeddingAverageLayer

The model below uses `EmbeddingAverageLayer` instead of `OneHotEncoderVectorized + LinearLayerAvgMixer`.
The architecture is identical in terms of what it learns where the input is now the raw integer context tensor rather than one-hot vectors.

In [ ]:
WEIGHTS_FN       = "/home/enarde/Documents/Design-Of-AI-Systems/miniproject/word2vec_ns_fast.pt"
LATENT_DIM_FAST  = 50
K_NEGATIVES_FAST = 5
LR               = 0.025   # original Word2Vec paper learning rate
N_EPOCHS         = 10

# Always create fresh model objects first
embedding_layer = EmbeddingAverageLayer(
    vocab_size=tokenizer.VOCAB_SIZE,
    embed_dim=LATENT_DIM_FAST,
    lr=LR
)
word2vecCBOW_fast = Model([embedding_layer], debug_mode=False)

ns_loss_fast = NegativeSamplingLoss(
    tokenizer=tokenizer,
    embed_dim=LATENT_DIM_FAST,
    k=K_NEGATIVES_FAST,
    lr=LR
)

weights_path = Path(WEIGHTS_FN)
if weights_path.exists():
    checkpoint = T.load(weights_path, weights_only=False)
    embedding_layer.weights          = checkpoint["input_embeddings"]
    ns_loss_fast.out_embeddings      = checkpoint["output_embeddings"]
    ns_history_fast                  = checkpoint["history"]
    print(f"Loaded weights from {WEIGHTS_FN}  (trained {len(ns_history_fast['train'])} epochs)")
else:
    ns_history_fast = train_ns(word2vecCBOW_fast, ns_loss_fast, dataloader_ns, dataloader_ns_val, N_EPOCHS)

    T.save({
        "input_embeddings":  embedding_layer.weights,
        "output_embeddings": ns_loss_fast.out_embeddings,
        "history":           ns_history_fast,
    }, weights_path)
    print(f"Saved weights to {WEIGHTS_FN}")

Epoch 1/10:  21%|█    | 609/2867 [00:10<00:39, 57.20it/s, train_loss=81048.1238]


KeyboardInterrupt: 

In [ ]:
epochs = range(1, len(ns_history_fast["train"]) + 1)

plt.figure(figsize=(8, 4))
plt.plot(epochs, ns_history_fast["train"], label="Train loss")
plt.plot(epochs, ns_history_fast["val"],   label="Val loss", linestyle="--")
plt.xlabel("Epoch")
plt.ylabel("Avg NS loss per sample")
plt.title("Word2Vec CBOW — Negative Sampling Training Curve")
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
def nearest_neighbors(word, embeddings, word_to_index, index_to_word, n=8):
    """Return the n most similar words by cosine similarity."""
    idx = word_to_index.get(word)
    if idx is None:
        return f"'{word}' not in vocabulary"
    vec = embeddings[idx]
    norms  = embeddings.norm(dim=1).clamp(min=1e-8)
    scores = (embeddings @ vec) / (norms * vec.norm().clamp(min=1e-8))
    top    = scores.argsort(descending=True)[1:n+1]   # skip self at rank 0
    return [(index_to_word[i.item()], round(scores[i].item(), 4)) for i in top]


# Probe a set of words
probe_words = ["time", "world", "war", "city", "year", "new", "first", "people",
               "government", "american", "great", "history"]

print(f"{'Word':>14}   Nearest neighbors (cosine similarity)\n{'─'*80}")
for word in probe_words:
    result = nearest_neighbors(word, embedding_layer.weights,
                               tokenizer.word_to_index, tokenizer.index_to_word)
    if isinstance(result, str):
        print(f"{word:>14}   {result}")
    else:
        nbrs = ",  ".join(f"{w} ({s:.3f})" for w, s in result)
        print(f"{word:>14}   {nbrs}")

# Visualizing the word embeddings

## Visualization methods

### PCA:
PCA is one of the oldest tricks in the book when it comes to reducing dimensionality of data in order to make it possible to visualize in a 2d, and sometimes 3d, space. Its foundational paper was released all the way back in 1933 by a Harald Hotelling, making it almost 100 years old.

PCA relies on two important concepts from the field of linear algebra, eigenvectors and matrix transformations (base changes to be more exact). The PCA methods works in several steps: 
- Initially the eigenvectors of the dataset features correlation matrix is computed, these eigenvalues can be interpreted as how much the covariance
$$\begin{pmatrix} 1 & \sigma^2_{12} & \sigma^2_{13} \\ \sigma^2_{12} & 1 & \sigma^2_{23} \\ \sigma^2_{13} & \sigma^2_{23} & 1 \end{pmatrix} \longrightarrow \boldsymbol{\lambda}_1 = \begin{pmatrix} \lambda_{11} \\ \lambda_{21} \\ \lambda_{31} \end{pmatrix}, \boldsymbol{\lambda}_2 = \begin{pmatrix} \lambda_{12} \\ \lambda_{22} \\ \lambda_{32} \end{pmatrix}, \boldsymbol{\lambda}_3 = \begin{pmatrix} \lambda_{13} \\ \lambda_{23} \\ \lambda_{33} \end{pmatrix}$$


### UMAP:

## Visualizing results

### PCA

In [ ]:
TOP_N = 150   # number of most-frequent words to plot

# Exclude PAD token, then take the top-N by training frequency
freq = tokenizer.word_freq.copy()
freq[tokenizer.PAD_INDEX] = 0
top_indices = np.argsort(freq)[::-1][:TOP_N]
top_words   = [tokenizer.index_to_word[i] for i in top_indices]

embeddings_np = embedding_layer.weights.detach().numpy()
top_vecs      = embeddings_np[top_indices]

pca     = PCA(n_components=2)
reduced = pca.fit_transform(top_vecs)
var_exp = pca.explained_variance_ratio_

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(reduced[:, 0], reduced[:, 1], s=12, alpha=0.6)
for i, word in enumerate(top_words):
    ax.annotate(word, (reduced[i, 0], reduced[i, 1]), fontsize=7, alpha=0.85)

ax.set_title(
    f"PCA of top-{TOP_N} word embeddings\n"
    f"PC1 explains {var_exp[0]*100:.1f}% variance, PC2 explains {var_exp[1]*100:.1f}% variance"
)
ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
plt.tight_layout()
plt.show()

### UMAP

In [ ]:
try:
    import umap
except ImportError:
    raise ImportError("Install with: pip install umap-learn")

TOP_N_UMAP = 150

freq = tokenizer.word_freq.copy()
freq[tokenizer.PAD_INDEX] = 0
top_indices = np.argsort(freq)[::-1][:TOP_N_UMAP]
top_words   = [tokenizer.index_to_word[i] for i in top_indices]

embeddings_np = embedding_layer.weights.detach().numpy()
top_vecs      = embeddings_np[top_indices]

reducer = umap.UMAP(n_components=2, n_neighbors=15, min_dist=0.1, random_state=42)
reduced = reducer.fit_transform(top_vecs)

fig, ax = plt.subplots(figsize=(14, 10))
ax.scatter(reduced[:, 0], reduced[:, 1], s=12, alpha=0.6)
for i, word in enumerate(top_words):
    ax.annotate(word, (reduced[i, 0], reduced[i, 1]), fontsize=7, alpha=0.85)

ax.set_title(f"UMAP of top-{TOP_N_UMAP} word embeddings  (n_neighbors=15, min_dist=0.1)")
ax.set_xlabel("UMAP 1")
ax.set_ylabel("UMAP 2")
plt.tight_layout()
plt.show()